# Files for IGV

One objective: turn the pipeline's outputs into sorted, compressed, indexed files that IGV can load: the predicted ORFs, the filtered transcripts, and (when you have them) the alignments.

In [1]:
import sys, glob, os
sys.path.insert(0, "src")
from igv_prepare import orf_index, write_specific_bed, index_annotation, psl_to_bed12, prepare_bam, check_index
from paths import run_locations

loc = run_locations()                   # every folder of the run, from paths.json
bam_dir = None                          # folder with the alignment BAM files, or None if you have none
out_dir = "igv"

**Explanation.** IGV reads large files fast only when they are sorted by position, compressed with bgzip and have an index file next to them (`.tbi` for annotation, `.bai` for BAM). The functions in `src/igv_prepare.py` do these steps. The run's folders come from `paths.json`, as in the other notebooks; `bam_dir` is the one setting here that depends on where your alignments are.

In [2]:
gtf = glob.glob(f"{loc['protein_classification']}/*collapsed_high_confidence_ORF.gtf")[0]
bed = glob.glob(f"{loc['protein_classification']}/*collapsed_high_confidence_ORF.bed")[0]
gtf_gz = index_annotation(gtf, out_dir, "gff")
bed_gz = index_annotation(bed, out_dir, "bed")
gtf_gz, bed_gz

('igv/SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.gtf.gz',
 'igv/SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.bed.gz')

**Explanation.** The predicted ORFs come as a GTF (genes and exons) and a BED (one line per ORF). Both are sorted, compressed and indexed into the `igv/` folder.

In [ ]:
hashids = glob.glob(f"{loc['run_dir']}/**/*collapsed_high_confidence_ORF_hashids_with_cpm.txt", recursive=True)[0]
de_orf = glob.glob(f"{loc['de_orf']}/*_DE_ORF_edgeR_results.txt")
idx = orf_index(gtf, hashids, de_orf[0] if de_orf else None)
idx.to_csv(f"{out_dir}/{loc['dataset_name']}.orf_index.tsv", sep="\t", index=False)
idx[idx["gene"] == "HLA-B"][["orf_base_id", "locus", "strand", "coding_exons", "n_isoforms"] + (["logFC", "FDR"] if de_orf else [])]

**Explanation.** IGV shows each ORF as one feature and cannot say which transcript isoforms it came from. This table has one row per predicted ORF: its gene, its `locus` (paste it into IGV's search box to go there), strand, number of coding exons, coding length, the ORF ratio, and every isoform that gives this ORF (`orf_all_isoform_id`). If the differential ORF expression table for the comparison is present, `logFC` and `FDR` are added, so you can go from a hit in the volcano plot to its place in the genome. It is a plain table, written to `igv/` with the other files; change `"HLA-B"` to look at another gene.

In [3]:
psl = glob.glob(f"{loc['leafcutter']}/*transcriptome.filtered.psl")[0]
os.makedirs(out_dir, exist_ok=True)
bed12 = f"{out_dir}/" + os.path.basename(psl).replace(".psl", ".bed")
n = psl_to_bed12(psl, bed12)
transcripts_gz = index_annotation(bed12, out_dir, "bed")
print(n, "filtered transcripts ->", transcripts_gz)

305 filtered transcripts -> igv/SERVES_LRP2_test.transcriptome.filtered.bed.gz


**Explanation.** The filtered transcript set comes as a PSL file, which cannot be indexed. It is turned into BED12 (a line per transcript with its exons), then sorted, compressed and indexed like the others. This is the track that shows every transcript the pipeline kept.

In [ ]:
import pandas as pd
inputs = {"isoforms": transcripts_gz, "orfs": bed, "variants": None}   # the transcript BED12 made above, the ORF BED, and (variants) the table itself
for level, bed_in in inputs.items():
    table = f"tables/{loc['comparison']}_specific_{level}.tsv"
    if not os.path.exists(table):
        print("no table yet for", level, "- run specific_variants.ipynb first"); continue
    plain = f"{out_dir}/{loc['comparison']}.specific_{level}.bed"
    n = write_specific_bed(table, plain, level, bed_in)
    print(n, "lines ->", index_annotation(plain, out_dir, "bed") if n else "nothing to index")

**Explanation.** The tables from `specific_variants.ipynb` list the features specific to each group. This writes them as IGV tracks, coloured by group with the same colours as the heatmaps: the specific transcript isoforms, the specific ORFs, and the specific splicing variants (each intron drawn as its own bar). They are sorted, compressed and indexed like the other files. Load them next to the whole-set tracks and the Refseq genes. The intron positions are the ones leafcutter reports and may be off by one base against the genome.

In [4]:
for f in sorted(glob.glob(f"{out_dir}/*.gz")):
    print(check_index(f), "records in", os.path.basename(f))

132 records on chr22 in SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.bed.gz
1619 records on chr22 in SERVES_LRP2_test.predicted_proteome.collapsed_high_confidence_ORF.gtf.gz
305 records on chr22 in SERVES_LRP2_test.transcriptome.filtered.bed.gz


**Explanation.** A check that each index works: asking for chr22 should return records. Change `chr22` to a chromosome in your data.

In [5]:
if bam_dir:
    for bam in sorted(glob.glob(f"{bam_dir}/*.bam")):
        print(prepare_bam(bam, out_dir))
else:
    print("bam_dir is not set: no alignments to prepare")

bam_dir is not set: no alignments to prepare


**Explanation.** Each alignment BAM is sorted by position and indexed (a `.bai` file), into the `igv/` folder. The test results contain no BAM files, so this cell does nothing until `bam_dir` points at a folder of them. In IGV use File, Load from File and pick the `.gz` and `.bam` files; IGV finds the index next to each.